# Data Understanding

Assumes the Sri Lanka load-forecasting CSV (`load_forecasting_dataset_corrected.csv`, Kaggle: `isuranga/load-forecasting-dataset`) has been placed in `data/raw/`.

**Data authenticity note:** the Kaggle listing describes this as CEB-sourced, but the checks in the "Data authenticity check" section below (demand magnitude, GDP scale, season labeling, missing-value rate) indicate it is synthetically generated rather than raw utility telemetry. It is used here as a synthetic Sri Lanka-style dataset for demonstrating the forecasting methodology, not as validated real-world CEB figures -- see proposal.txt Sections 7.1 and 28.

In [ ]:
import os
import sys
from pathlib import Path

import yaml

# Works both in the local repo and on Kaggle. Locally, config.yaml is read
# from disk. On Kaggle there is no repo checkout -- only whatever cells you
# paste -- so the config is built inline instead, pointed at the attached
# dataset under /kaggle/input. IS_KAGGLE is detected via KAGGLE_KERNEL_RUN_TYPE
# (always set on Kaggle) rather than checking /kaggle/input directly, since
# that directory doesn't exist until a dataset is attached -- checking it
# directly would silently fall through to the local branch and fail with a
# confusing "config.yaml not found" instead. The dataset is then located by
# searching /kaggle/input for the raw CSV by name rather than assuming a
# fixed mount depth, since Kaggle has been observed to mount an attached
# dataset at different depths depending on how it's attached (flat
# /kaggle/input/<slug>/... vs nested /kaggle/input/datasets/<owner>/<slug>/...).
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    RAW_FILENAME = "load_forecasting_dataset_corrected.csv"
    matches = list(Path("/kaggle/input").rglob(RAW_FILENAME))
    if not matches:
        raise RuntimeError(
            f"Could not find {RAW_FILENAME} anywhere under /kaggle/input -- "
            "attach the dataset containing it via the notebook's Data panel."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Found {len(matches)} copies of {RAW_FILENAME} under /kaggle/input: "
            f"{matches} -- remove the extras or point RAW_DIR at the right one manually."
        )
    print("Found raw data at:", matches[0])

    REPO_ROOT = Path("/kaggle/working")
    config = {
        "data": {
            "raw_path": str(matches[0].parent),
            "processed_path": "/kaggle/working/data/processed",
            "processed_file": "/kaggle/working/data/processed/demand_clean.csv",
            "features_file": "/kaggle/working/data/processed/features.csv",
            "timestamp_col": "timestamp",
            "target_col": "demand",
            "frequency": "h",
        },
        "peak_demand": {"percentile_threshold": 0.95, "seasonal": True},
    }
else:
    def find_repo_root(start: Path) -> Path:
        for parent in [start, *start.parents]:
            if (parent / "config.yaml").exists():
                return parent
        raise FileNotFoundError("config.yaml not found in any parent directory")

    REPO_ROOT = find_repo_root(Path.cwd())
    sys.path.insert(0, str(REPO_ROOT))

    with open(REPO_ROOT / "config.yaml") as f:
        config = yaml.safe_load(f)

config

## Load raw data

In [6]:
import pandas as pd

RAW_DIR = REPO_ROOT / config["data"]["raw_path"]
RAW_FILE = "load_forecasting_dataset_corrected.csv"

df = pd.read_csv(RAW_DIR / RAW_FILE)
print(f"{RAW_FILE}:", df.shape)

load_forecasting_dataset_corrected.csv: (189888, 15)


## Column names and data types

In [7]:
df.dtypes

Timestamp                       object
Temperature (°C)               float64
Humidity (%)                   float64
Wind Speed (m/s)               float64
Rainfall (mm)                  float64
Solar Irradiance (W/m²)        float64
GDP (LKR)                      float64
Per Capita Energy Use (kWh)    float64
Electricity Price (LKR/kWh)    float64
Day of Week                      int64
Hour of Day                      int64
Month                            int64
Season                          object
Public Event                     int64
Load Demand (kW)               float64
dtype: object

## Sample observations

In [8]:
df.head()

,Timestamp,Temperature (°C),Humidity (%),Wind Speed (m/s),Rainfall (mm),Solar Irradiance (W/m²),GDP (LKR),Per Capita Energy Use (kWh),Electricity Price (LKR/kWh),Day of Week,Hour of Day,Month,Season,Public Event,Load Demand (kW)
0,1/1/2020 0:00,28.993428,75.011269,1.053861,4.140513,185.892561,925.621430,502.915605,20.454440,2,0,1,Summer,0,1599.342831
1,1/1/2020 0:15,27.723471,77.024015,1.085152,9.446997,281.782650,1020.823521,497.286366,27.776449,2,0,1,Summer,0,1472.347140
2,1/1/2020 0:30,29.295377,74.732958,3.363800,4.265813,328.942058,1028.847455,488.816292,21.097420,2,0,1,Summer,0,1629.537708
3,1/1/2020 0:45,31.046060,87.615995,2.539148,1.038103,336.407064,937.963002,468.038834,26.032137,2,0,1,Summer,1,1804.605971
4,1/1/2020 1:00,27.531693,79.709858,1.366819,4.201393,205.494256,934.477462,488.565716,27.079114,2,1,1,Summer,0,1453.169325


## Time period and observation frequency

In [9]:
timestamps = pd.to_datetime(df["Timestamp"]).sort_values()

print("Time range:", timestamps.min(), "to", timestamps.max())
print("Number of unique timestamps:", timestamps.nunique())

inferred_freq = pd.infer_freq(timestamps.iloc[:100])
print("Inferred frequency (first 100 timestamps):", inferred_freq)

Time range: 2020-01-01 00:00:00 to 2025-05-31 23:45:00
Number of unique timestamps: 189888
Inferred frequency (first 100 timestamps): 15min


## Target variable — `Load Demand (kW)`

In [10]:
target_col_raw = "Load Demand (kW)"

print(df[target_col_raw].describe())
print()
print("Missing values:", df[target_col_raw].isna().sum(),
      f"({df[target_col_raw].isna().mean():.2%})")

count    189888.000000
mean       1500.154422
std         199.926679
min         606.879227
25%        1365.223394
50%        1500.274982
75%        1635.071296
max        2412.422945
Name: Load Demand (kW), dtype: float64

Missing values: 0 (0.00%)


## Missing values overview

In [11]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

Series([], dtype: int64)

## Data authenticity check

The Kaggle listing describes this dataset as CEB-sourced. The checks below suggest otherwise -- see proposal.txt Sections 7.1 and 28 for how this is framed in the write-up.

In [12]:
print("Demand (kW) -- Sri Lanka's real national peak grid demand is ~2,500-2,700 MW:")
print(df[target_col_raw].describe()[["mean", "min", "max"]])
print()

print("GDP (LKR) column -- real Sri Lankan nominal GDP is in the tens of trillions of LKR:")
print(df["GDP (LKR)"].describe()[["mean", "min", "max"]])
print()

print("Season vs Month -- does not match a tropical/monsoon climate or any standard hemisphere mapping:")
print(pd.crosstab(df["Month"], df["Season"]))
print()

print("Duplicate timestamps:", pd.to_datetime(df["Timestamp"]).duplicated().sum())
print("Total missing values across all columns:", df.isna().sum().sum())

Demand (kW) -- Sri Lanka's real national peak grid demand is ~2,500-2,700 MW:
mean    1500.154422
min      606.879227
max     2412.422945
Name: Load Demand (kW), dtype: float64

GDP (LKR) column -- real Sri Lankan nominal GDP is in the tens of trillions of LKR:
mean    1000.083804
min      779.558268
max     1251.690264
Name: GDP (LKR), dtype: float64

Season vs Month -- does not match a tropical/monsoon climate or any standard hemisphere mapping:
Season   Fall  Summer  Winter
Month                        
1           0   17856       0
2           0   16320       0
3       17856       0       0
4       17280       0       0
5       17856       0       0
6           0       0   14400
7           0       0   14880
8           0       0   14880
9       14400       0       0
10      14880       0       0
11      14400       0       0
12          0   14880       0

Duplicate timestamps: 0
Total missing values across all columns: 0
